In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['CLV'] = df['MonthlyCharges'] * df['tenure']


In [ ]:
# Q21: Churn prediction feature dataset using Pandas feature engineering
features = pd.DataFrame()

features['customerID']      = df['customerID']
features['tenure']          = df['tenure']
features['monthly_charges'] = df['MonthlyCharges']
features['total_charges']   = df['TotalCharges']
features['clv']             = df['CLV']

features['is_senior']       = df['SeniorCitizen']
features['has_partner']     = (df['Partner'] == 'Yes').astype(int)
features['has_dependents']  = (df['Dependents'] == 'Yes').astype(int)
features['has_phone']       = (df['PhoneService'] == 'Yes').astype(int)
features['has_security']    = (df['OnlineSecurity'] == 'Yes').astype(int)
features['has_techsupport'] = (df['TechSupport'] == 'Yes').astype(int)
features['is_paperless']    = (df['PaperlessBilling'] == 'Yes').astype(int)

features['contract_monthly'] = (df['Contract'] == 'Month-to-month').astype(int)
features['contract_yearly']  = (df['Contract'] == 'One year').astype(int)

features['risk_score'] = features.apply(lambda row:
    (3 if df.loc[row.name, 'Contract'] == 'Month-to-month' else 1 if df.loc[row.name, 'Contract'] == 'One year' else 0) +
    (3 if row['tenure'] <= 12 else 2 if row['tenure'] <= 24 else 1 if row['tenure'] <= 48 else 0) +
    (2 if df.loc[row.name, 'TechSupport'] == 'No' else 0) +
    (2 if df.loc[row.name, 'OnlineSecurity'] == 'No' else 0), axis=1)

features['churn'] = (df['Churn'] == 'Yes').astype(int)

features.head()

In [ ]:
# Q23: Retention dashboard dataset with retention KPIs
retention = {}

retention['total_customers']      = len(df)
retention['churned_customers']    = (df['Churn'] == 'Yes').sum()
retention['retained_customers']   = (df['Churn'] == 'No').sum()
retention['churn_rate']           = round((df['Churn'] == 'Yes').sum() / len(df) * 100, 2)
retention['retention_rate']       = round((df['Churn'] == 'No').sum() / len(df) * 100, 2)
retention['monthly_revenue']      = round(df['MonthlyCharges'].sum(), 2)
retention['monthly_revenue_lost'] = round(df[df['Churn'] == 'Yes']['MonthlyCharges'].sum(), 2)
retention['annual_revenue_lost']  = round(retention['monthly_revenue_lost'] * 12, 2)
retention['avg_clv_retained']     = round(df[df['Churn'] == 'No']['CLV'].mean(), 2)
retention['avg_clv_churned']      = round(df[df['Churn'] == 'Yes']['CLV'].mean(), 2)

for key, value in retention.items():
    print(f'{key:35s}: {value}')

In [ ]:
# Q24: Rule-based churn detection engine
def churn_risk_level(row):
    score = 0

    if row['Contract'] == 'Month-to-month':
        score += 3
    elif row['Contract'] == 'One year':
        score += 1

    if row['tenure'] <= 12:
        score += 3
    elif row['tenure'] <= 24:
        score += 2
    elif row['tenure'] <= 48:
        score += 1

    if row['OnlineSecurity'] == 'No':
        score += 2
    if row['TechSupport'] == 'No':
        score += 2

    if row['PaymentMethod'] == 'Electronic check':
        score += 1

    if row['SeniorCitizen'] == 1:
        score += 1

    if score >= 9:
        return 'Critical'
    elif score >= 6:
        return 'High'
    elif score >= 3:
        return 'Medium'
    else:
        return 'Low'

df['churn_risk'] = df.apply(churn_risk_level, axis=1)

df.groupby('churn_risk').agg(
    customer_count = ('customerID', 'count'),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2))
).reset_index().sort_values('churn_rate', ascending=False)